In [ ]:
import base64
import requests
import time
import pandas as pd
import os

# OpenAI API Key - Replace with your GPT API key
api_key = " "

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Read CSV
questions_df = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs _Bad Prompt/VLAT Questions.csv")
questions = questions_df.to_dict('records')
responses_data = []


for i, question in enumerate(questions, start=1):
    time.sleep(5)
    try:
        print(f"\nProcessing question {i}:")
        print(question)
        
        # Path to your image
        image_path = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/" + str(question.get('vis', '')) + ".png"
        
        # Question text
        question_text = question.get('question: ', '')
        question_options = question.get('option:', '')
        correct_ans = str(question.get('correct', '')).strip()
        
        print(f"Processing image: {image_path}")
        print(f"Question: {question_text}")
        print(f"Options: {question_options}")
        print(f"Correct answer: {correct_ans}")
        
        # Getting the base64 string
        base64_image = encode_image(image_path)
        
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }
        
        payload = {
            "model": "gpt-4.5-preview-2025-02-27", # Use GPT-4o for vision capabilities
            "max_tokens": 5000,
            "temperature": 0.0,
            "messages": [
                {
                    "role": "system",
                    "content": "You are an assistant, skilled in reading and interpreting visually represented data."
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": 'I am about to show you an image and ask you a multiple choice question about that image. Select the BEST answer, based only on the chart and not external knowledge.' +
                                    'End with: "Correct Answer: ". Just write the value, nothing else. Do not write anything after this. \n\n' +
                                   '\n\n' + question_text + " " + question_options
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]
        }
        
        # Add retry logic for overloaded errors
        max_retries = 3
        retry_delay = 20  # seconds
        
        for retry in range(max_retries):
            try:
                time_start = time.perf_counter()
                response_raw = requests.post(
                    "https://api.openai.com/v1/chat/completions",
                    headers=headers,
                    json=payload
                )
                time_end = time.perf_counter()
                
                response = response_raw.json()
                
                # If we get a rate limit error, wait and retry
                if 'error' in response and response['error'].get('code') == 'rate_limit_exceeded':
                    if retry < max_retries - 1:  # Don't sleep on the last retry
                        print(f"\nAPI rate limited, waiting {retry_delay} seconds before retry {retry + 1}/{max_retries}")
                        time.sleep(retry_delay)
                        continue
                
                break  # If we get here, we either got a good response or a different error
                
            except Exception as e:
                print(f"Error during API call (attempt {retry + 1}/{max_retries}):", str(e))
                if retry < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                raise  # Re-raise the last exception if we've exhausted all retries
        print("\nAPI Response:", response)  # Debug print
        
        if 'error' in response:
            gpt_answer = f"Error: {response['error'].get('message', 'Unknown error')}"
            is_correct = "N/A"
        else:
            # Extract the answer after "Correct Answer: "
            full_response = response["choices"][0]["message"]["content"].strip()
            if "Correct Answer: " in full_response:
                gpt_answer = full_response.split("Correct Answer: ")[-1].strip()
                # Case-insensitive comparison after stripping whitespace
                is_correct = gpt_answer.strip().upper() == correct_ans.strip().upper()
            else:
                gpt_answer = "Error: No answer in correct format"
                is_correct = "N/A"
        
        responses_data.append([gpt_answer, time_end-time_start, is_correct])
        print(f"\nAnswer: {gpt_answer}")
        print(f"Time taken: {time_end-time_start:.2f} seconds")
        print(f"Correct? {is_correct}")
        
        # Increase delay between requests to 15 seconds
        time.sleep(max(15 - (time_end-time_start), 0))
        
    except Exception as e:
        print(f"Error processing question {i}:", str(e))
        responses_data.append([f"Error: {str(e)}", 0, "N/A"])

# Create Results directory if it doesn't exist
results_dir = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs _Bad Prompt/Results/2_GPT 4.5_Final Results with Explanation/"
os.makedirs(results_dir, exist_ok=True)

results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
results_df.to_csv(results_dir + "GPT_VLAT_" + str(int(time.time())) + ".csv", index_label="id")
print("\n*** Finished ***")


Processing question 1:
{'id': 1, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_1', 'question: ': 'What was the price of a barrel of oil in February 2015? ', 'option:': '$39.72; $57.36; $48.90; $62.85', 'correct': '$48.90 '}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: What was the price of a barrel of oil in February 2015? 
Options: $39.72; $57.36; $48.90; $62.85
Correct answer: $48.90

API Response: {'id': 'chatcmpl-B9LCr4LFFeVw886l6RVIeRCkWocVM', 'object': 'chat.completion', 'created': 1741566169, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Correct Answer: $48.90', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 542, 'completion_tokens': 8, 'total_tokens': 550, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rej


Processing question 8:
{'id': 8, 'dropped': 'no', 'vis': 'VLAT_b', 'item': 'b_3', 'question: ': 'What is the range of the average internet speed in Asia?', 'option:': '0 - 25 Mbps; 9 - 18 Mbps; 3 - 20 Mbps; 4 - 7 Mbps', 'correct': '3 - 20 Mbps'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_b.png
Question: What is the range of the average internet speed in Asia?
Options: 0 - 25 Mbps; 9 - 18 Mbps; 3 - 20 Mbps; 4 - 7 Mbps
Correct answer: 3 - 20 Mbps

API Response: {'id': 'chatcmpl-B9LF75uIBrotFo0BoP1gTmFb29nna', 'object': 'chat.completion', 'created': 1741566309, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Correct Answer: 0 - 25 Mbps', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 546, 'completion_tokens': 10, 'total_tokens': 556, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'au


Processing question 15:
{'id': 15, 'dropped': 'no', 'vis': 'VLAT_c', 'item': 'c_6', 'question: ': 'The ratio of the cost of Soda to the cost of Water in Orlando is higher than that of Washington D.C.', 'option:': 'True; False', 'correct': 'TRUE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_c.png
Question: The ratio of the cost of Soda to the cost of Water in Orlando is higher than that of Washington D.C.
Options: True; False
Correct answer: TRUE

API Response: {'id': 'chatcmpl-B9LHPpRT6vk6E57OEOiWlc3nXbvFA', 'object': 'chat.completion', 'created': 1741566451, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'False\nCorrect Answer: False', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 533, 'completion_tokens': 7, 'total_tokens': 540, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'au


Processing question 22:
{'id': 22, 'dropped': 'no', 'vis': 'VLAT_f', 'item': 'f_1', 'question: ': 'How many people have rated the taxi between 4.0 and 4.2?', 'option:': '145; 183; 250; 310', 'correct': '310'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_f.png
Question: How many people have rated the taxi between 4.0 and 4.2?
Options: 145; 183; 250; 310
Correct answer: 310

API Response: {'id': 'chatcmpl-B9LJdYyuA074LLJy1b1blGsnGcaUq', 'object': 'chat.completion', 'created': 1741566589, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Correct Answer: 310', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 876, 'completion_tokens': 6, 'total_tokens': 882, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, '


API Response: {'id': 'chatcmpl-B9LLt3QkLRWrwfxYVjpib5HG5h97e', 'object': 'chat.completion', 'created': 1741566729, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Correct Answer: 53.9 - 123.6 kg', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 558, 'completion_tokens': 14, 'total_tokens': 572, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': None}

Answer: 53.9 - 123.6 kg
Time taken: 3.83 seconds
Correct? False

Processing question 30:
{'id': 30, 'dropped': 'yes', 'vis': 'VLAT_g', 'item': 'g_4', 'question: ': 'More than 60 percent of males lie between the weights of 60 kg and 100 kg.', 'option:': 'True; False', 'correct': 'TRUE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_


Processing question 37:
{'id': 37, 'dropped': 'no', 'vis': 'VLAT_i', 'item': 'i_3', 'question: ': 'What was the range of the average price of a pound of coffee beans between January 2013 and December 2014?', 'option:': '$5.4 - $6.2; $4.6 - $5.9; $4.6 - $6.0; $4.6 - $6.1', 'correct': '$4.6 - $6.1'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_i.png
Question: What was the range of the average price of a pound of coffee beans between January 2013 and December 2014?
Options: $5.4 - $6.2; $4.6 - $5.9; $4.6 - $6.0; $4.6 - $6.1
Correct answer: $4.6 - $6.1

API Response: {'id': 'chatcmpl-B9LOTkzlrWsegzuK1rmyLKpzH5y8w', 'object': 'chat.completion', 'created': 1741566889, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Correct Answer: $4.6 - $6.1', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 571, 'completion_tokens': 13, 'total_tokens': 584, 'prompt_tokens_det


Processing question 44:
{'id': 44, 'dropped': 'no', 'vis': 'VLAT_j', 'item': 'j_5', 'question: ': 'The number of girls named ‘Isla’ was __________ from 2009 to 2012.', 'option:': 'rising; falling; staying', 'correct': 'rising'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_j.png
Question: The number of girls named ‘Isla’ was __________ from 2009 to 2012.
Options: rising; falling; staying
Correct answer: rising

API Response: {'id': 'chatcmpl-B9LQj4m93PD7wZiOVZNPPRYeeg3Ht', 'object': 'chat.completion', 'created': 1741567029, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'rising\n\nCorrect Answer: rising', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 532, 'completion_tokens': 8, 'total_tokens': 540, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_predicti


API Response: {'id': 'chatcmpl-B9LSzqsTOUcdCaMlB1ElUHlTLw4za', 'object': 'chat.completion', 'created': 1741567169, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Correct Answer: G', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 876, 'completion_tokens': 5, 'total_tokens': 881, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': None}

Answer: G
Time taken: 3.57 seconds
Correct? True

Processing question 52:
{'id': 52, 'dropped': 'no', 'vis': 'VLAT_h', 'item': 'h_6', 'question: ': 'A group of the metro systems of the world has approximately 300 stations and around a 200 km system length.', 'option:': 'True; False', 'correct': 'FALSE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/N


API Response: {'id': 'chatcmpl-B9LVFLyraKHG6X8yNrsW69lxzKO7a', 'object': 'chat.completion', 'created': 1741567309, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Correct Answer: 20%', 'refusal': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 886, 'completion_tokens': 7, 'total_tokens': 893, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': None}

Answer: 20%
Time taken: 3.72 seconds
Correct? False

Processing question 59:
{'id': 59, 'dropped': 'no', 'vis': 'VLAT_l', 'item': 'l_2', 'question: ': 'For which website was the number of unique visitors the largest in 2010?', 'option:': 'Facebook; Amazon; Bing; Google', 'correct': 'Google'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Gr